# Notebook 05: Hensen Open Data Audit

This notebook performs a detailed audit of the Hensen et al. (2015) Delft loophole-free Bell-test dataset. 
It verifies the loading process, filtering steps, and reproduction of the published CHSH result.

In [ ]:
from __future__ import annotations
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path

# Standardized project root addition
project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from xtheta.data.adapters.hensen import load_hensen_dataset
from xtheta.data.validation import run_open_data_chsh_validation

## 1. Load Raw Data

We load the raw data to see the initial row count and inspect the first few lines.

In [ ]:
data_path = project_root / "data" / "open_bell" / "hensen" / "raw" / "bell_open_data.txt"
df_raw = pd.read_csv(data_path, header=None)
print(f"Raw row count: {len(df_raw)}")
df_raw.head()

## 2. Apply Hensen Adapter

The adapter applies official filtering and mapping logic.

In [ ]:
data_iterator = load_hensen_dataset(str(data_path))
df_filtered = pd.concat(list(data_iterator))
print(f"Valid Bell trial count: {len(df_filtered)}")

## 3. Setting Pair Distribution

The 245 trials should be distributed across the four setting pairs (00, 01, 10, 11).

In [ ]:
counts = df_filtered.groupby(['alice_setting', 'bob_setting']).size().reset_index(name='count')
print(counts)

## 4. Run CHSH Validation

Verify the CHSH S-value matches the target $S \approx 2.42$.

In [ ]:
results = run_open_data_chsh_validation(
    load_hensen_dataset(str(data_path)),
    dataset_name="hensen_audit",
    output_dir="../outputs/hensen_audit",
    bootstrap_samples=1000
)

## 5. Conclusion

Target S: 2.42 ± 0.20.  
Observed S: {results['CHSH_S']:.4f} ± {results['CHSH_S_se']:.4f}.  
Valid Trials: {results['row_count']}.  

The reproduction is considered successful if valid trials $\approx 245$ and S is within tolerance.